In [ ]:
# Tested using this python environment:

import numpy as np
import uproot

# Batch run_subrun.txt creation for all data and EXT files

The cells below generate a `run_subrun_lists/<label>_run_subrun.txt` file for every beam-on data and beam-off (EXT) file currently used by `create_df.py`. These get copied to `/exp/uboone` and fed to Zarko's `getDataInfo.py` (commands printed below), and the text outputs get pasted back into this notebook.

From the `getDataInfo.py -v3` output:
- **beam-on data POT** = `tor875_wcut`
- **beam-on data num triggers** = `E1DCNT_wcut`
- **EXT num triggers** = `EXT` (when run on an EXT file's run/subrun list)

These then get hard-coded into `_get_file_metadata` in `create_df.py`, with each EXT file's POT-equivalent computed as `data_pot * ext_num_triggers / data_num_triggers` using the matching run period's data numbers.

In [ ]:
import os
import sys
sys.path.append("../src")
from file_locations import data_files_location

# every beam-on data and beam-off (EXT) file we need POT / trigger numbers for,
# label -> filename in data_files_location
run_subrun_file_dic = {
    # beam-on data: POT from tor875_wcut, num triggers from E1DCNT_wcut
    "run1_open_data":  "checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_on_data_surprise_reco2_hist_1_5e19opendata.root",
    "run3_open_data":  "checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_on_data_surprise_reco2_hist_3_1e19opendata.root",
    "run4a_open_data": "checkout_MCC9.10_Run4a_BNB_beam_on_data_surprise_reco2_hist_opendata_19550.root",
    "run4b_open_data": "checkout_MCC9.10_Run4b_v10_04_07_20_BNB_beam_on_metapatch_retuple_retuple_hist_opendata_20700.root",
    # beam-off EXT: num triggers from EXT
    "run1_ext":        "checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_off_data_surprise_reco2_hist_1.root",
    "run2_ext":        "checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_off_data_surprise_reco2_hist_2.root",
    "run3_ext":        "checkout_MCC9.10_Run123_v10_04_07_20_BNB_beam_off_data_surprise_reco2_hist_3.root",
    "run4a_ext":       "checkout_MCC9.10_Run4a_BNB_beam_off_data_surprise_reco2_hist.root",
    "run4b_ext":       "checkout_MCC9.10_Run4b_v10_04_07_20_BNB_beam_off_metapatch_retuple_retuple_hist.root",
    "run4c_ext":       "checkout_MCC9.10_Run4acd5_v10_04_07_14_BNB_beam_off_surprise_reco2_hist_4c.root",
    "run4d_ext":       "checkout_MCC9.10_Run4acd5_v10_04_07_14_BNB_beam_off_surprise_reco2_hist_4d.root",
    "run5_ext":        "checkout_MCC9.10_Run4acd5_v10_04_07_14_BNB_beam_off_surprise_reco2_hist_5.root",
}

os.makedirs("run_subrun_lists", exist_ok=True)

for label, filename in run_subrun_file_dic.items():
    f = uproot.open(f"{data_files_location}/{filename}")
    runs = f["nuselection"]["SubRun"]["run"].array(library="np")
    subruns = f["nuselection"]["SubRun"]["subRun"].array(library="np")
    f.close()

    outpath = f"run_subrun_lists/{label}_run_subrun.txt"
    with open(outpath, "w") as outfile:
        for r, s in zip(runs, subruns):
            outfile.write(f"{r} {s}\n")

    n_unique = len(set(zip(runs.tolist(), subruns.tolist())))
    dup_note = "" if n_unique == len(runs) else f"  <-- {len(runs) - n_unique} duplicate run/subrun pairs!"
    print(f"{label:<16} {len(runs):>7} lines ({n_unique:>7} unique run/subrun pairs) -> {outpath}{dup_note}")

In [ ]:
# commands to run after copying run_subrun_lists/ to /exp/uboone
# (scp the whole run_subrun_lists directory, then run these inside the apptainer)

print("/cvmfs/uboone.opensciencegrid.org/bin/shell_apptainer.sh")
print("source /cvmfs/uboone.opensciencegrid.org/products/setup_uboone_mcc9.sh;")
print("setup uboonecode v08_00_00_93 -q e17:prof;")
print()
for label in run_subrun_file_dic:
    print(f"/exp/uboone/app/users/zarko/getDataInfo.py -v3 --run-subrun-list {label}_run_subrun.txt | tee {label}_data_info.txt")

In [ ]:
# paste the getDataInfo.py text outputs here, one block per file
# (the next cell parses this string automatically)

getDataInfo_outputs = """
run1_open_data:
Read 14187 lines from run1_open_data_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
     3211774.0     6642299.0     6644787.0     2.845e+19     2.842e+19     6247112.0     2.811e+19     2.807e+19

run3_open_data:
Read 11578 lines from run3_open_data_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
     5385551.0     2581765.0     2591937.0     1.015e+19     1.018e+19     2301101.0      9.45e+18     9.471e+18

run4a_open_data:
Read 36279 lines from run4a_open_data_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
    15977636.0     5604801.0     5606979.0     2.144e+19     2.144e+19     4836758.0     2.098e+19     2.098e+19

run4b_open_data:
Read 43870 lines from run4b_open_data_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
    19928699.0     9676185.0     9677283.0     4.143e+19     4.143e+19     9218529.0     4.038e+19     4.038e+19

    
run1_ext:
Read 95517 lines from run1_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
    49551151.0    24050332.0    25644343.0     1.014e+20     1.013e+20    22263833.0     1.004e+20     1.002e+20

run2_ext:
Read 291072 lines from run2_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
   152116269.0    67659655.0    68377443.0     2.657e+20     2.654e+20    61494447.0     2.598e+20     2.595e+20
Warning!! BNB data for some of the requested runs/subruns is not in the database.
1 runs missing BNB data (number of subruns missing the data): 8883 (1),


run3_ext:
Read 288973 lines from run3_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
   143426455.0    60098992.0    60290143.0     2.303e+20     2.309e+20    51121153.0     2.125e+20     2.129e+20
Warning!! BNB data for some of the requested runs/subruns is not in the database.
1 runs missing BNB data (number of subruns missing the data): 16228 (1),


run4a_ext:
Read 62246 lines from run4a_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
    27940007.0    11469195.0    11472897.0     4.686e+19     4.685e+19     9755645.0     4.433e+19     4.432e+19

run4b_ext:
Read 194451 lines from run4b_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
    89010180.0    34265414.0    34146152.0     1.392e+20     1.392e+20    32096210.0     1.353e+20     1.353e+20

run4c_ext:
Read 112469 lines from run4c_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
    53659787.0    22536260.0    21744302.0     9.221e+19     9.225e+19    20336851.0     8.971e+19     8.975e+19

run4d_ext:
Read 129917 lines from run4d_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
    76563108.0    12166325.0    12251053.0     5.081e+19     5.085e+19    11178060.0     4.918e+19     4.922e+19

run5_ext:
Read 233954 lines from run5_ext_run_subrun.txt
           EXT         Gate2        E1DCNT        tor860        tor875   E1DCNT_wcut   tor860_wcut   tor875_wcut
   111457148.0    40469702.0    39736810.0     1.536e+20     1.538e+20    34990226.0     1.463e+20     1.465e+20
"""

In [ ]:
import re

# parse the pasted getDataInfo.py outputs above and print the numbers to hard-code in create_df.py
# data: pot = tor875_wcut, num_triggers = E1DCNT_wcut
# ext:  num_triggers = EXT

def parse_getDataInfo_outputs(text, expected_labels):
    parsed = {}
    current_label = None
    header_cols = None
    for line in text.splitlines():
        tokens = line.strip().split()
        label_match = re.match(r"^(\w+):$", line.strip())
        if label_match and label_match.group(1) in expected_labels:
            current_label = label_match.group(1)
            header_cols = None
            continue
        if current_label is None:
            continue
        if "tor875_wcut" in tokens:  # the header line
            header_cols = tokens
            continue
        if header_cols is not None and len(tokens) == len(header_cols):
            try:
                values = [float(t) for t in tokens]
            except ValueError:
                continue
            parsed[current_label] = dict(zip(header_cols, values))
            current_label = None
            header_cols = None
    return parsed

parsed = parse_getDataInfo_outputs(getDataInfo_outputs, set(run_subrun_file_dic))

missing = [label for label in run_subrun_file_dic if label not in parsed]
if missing:
    print(f"still missing getDataInfo outputs for: {missing}")
    print()

open_data_pot = {}
open_data_num_triggers = {}
ext_num_triggers = {}
for label, info in parsed.items():
    period = label.replace("run", "").replace("_open_data", "").replace("_ext", "")
    if label.endswith("_open_data"):
        open_data_pot[period] = info["tor875_wcut"]
        open_data_num_triggers[period] = int(info["E1DCNT_wcut"])
    else:
        ext_num_triggers[period] = int(info["EXT"])

print("# beam-on data (POT = tor875_wcut, num triggers = E1DCNT_wcut):")
for period in sorted(open_data_pot):
    print(f"run{period}_open_data_POT = {open_data_pot[period]:.4e}")
    print(f"run{period}_open_data_num_triggers = {open_data_num_triggers[period]}")
print()

print("# EXT (num triggers = EXT):")
for period in sorted(ext_num_triggers):
    print(f"run{period}_ext_num_triggers = {ext_num_triggers[period]}")
print()

# EXT POT-equivalent uses the matching run period's data POT/trigger ratio.
# Run 2 has no open data, so use the run 3 ratio (run 2 EXT is paired with run 3
# open data in the runs 1-5 open data weighting anyway). Runs 4c/4d/5 use the
# run 4b ratio (paired with run 4b open data in that same weighting).
ext_data_period_map = {"1": "1", "2": "3", "3": "3", "4a": "4a", "4b": "4b", "4c": "4b", "4d": "4b", "5": "4b"}

print("# EXT POT-equivalents (ext_num_triggers * data_pot / data_num_triggers):")
for ext_period, data_period in ext_data_period_map.items():
    if ext_period not in ext_num_triggers or data_period not in open_data_pot:
        print(f"# run {ext_period} ext: missing inputs, skipped")
        continue
    pot_per_trigger = open_data_pot[data_period] / open_data_num_triggers[data_period]
    ext_pot = ext_num_triggers[ext_period] * pot_per_trigger
    print(f"run{ext_period}_ext_pot_equivalent = {ext_pot:.4e}  # using run {data_period} POT/trigger ratio")